In [4]:
import os 
import re
import json 
import uuid 
from dotenv import load_dotenv
from pprint import pprint 
from typing import List, Dict, TypedDict, Literal, Optional 
from langchain_ollama import ChatOllama

load_dotenv()


True

### Config parameters 

In [5]:
config = {
    "data_cache_dir": "../artifacts/data",
    "vector_store_dir": "../artifacts/vector_store",
    "max_reasoning_iterations": 7,
    "top_k_retrieval": 10,
    "top_n_rerank": 3
}
os.makedirs(config['data_cache_dir'], exist_ok=True)
pprint(config)

{'data_cache_dir': '../artifacts/data',
 'max_reasoning_iterations': 7,
 'top_k_retrieval': 10,
 'top_n_rerank': 3,
 'vector_store_dir': '../artifacts/vector_store'}


### Instantiate models

In [6]:
deep_thinking_llm = ChatOllama(
    model='qwen3:32b',
    temperature=0.0,
    top_k=5,
    top_p=0.9,
    verbose=True,
    base_url="http://localhost:8080"
)

quick_thinking_llm = ChatOllama(
    model='qwen3:8b',
    temperature=0.0,
    top_k=5,
    top_p=0.9,
    verbose=True,
    base_url="http://localhost:8080"
)


### Agent State

#### InvestDebateState
- bull_history: arguments made by the bull agent 
- bear_history: arguments made by the bear agent 
- history: the full transcript of the debate 
- current_response: the most recent argument made 
- judge_decision: the manager's final decision 
- count: a counter to track the number of debate rounds 

#### RiskDebateState
- risky_history: history of the aggressive risk_taker 
- safe_history: history of the conservative agent 
- neutral_history: history of the balanced agent 
- history: full transcript of the risk discussion 
- latest_speaker: tracks the last agent to speak
- judge_decision: the portfolio manager's final decision 
- count: counter for risk dicussion rounds 

#### AgentState 
- company_of_interest: the stock ticker we are analyzing 
- trade_date: the date for the analysis 
- sender: tracks which agent last modified the state 
- investment_plan: the plan from the research manager 
- trader_investment_plan: the actionable plan from the Trader 
- final_trade_decision: the final decision from the portfolio manager

In [7]:
from typing import Annotated, Sequence, List 
from typing_extensions import TypedDict 
from langgraph.graph import MessagesState

class InvestDebateState(TypedDict):
    bull_history: str  
    bear_history: str 
    history: str 
    current_response: str 
    judge_decision: str 
    count: int

In [8]:
class RiskDebateState(TypedDict):
    risk_history: str 
    sage_history: str
    neutral_history: str
    history: str
    latest_speaker: str 
    current_risky_response: str 
    current_sage_response:str
    current_neutral_response:str
    judge_decision:str
    count:int


In [9]:
class AgentState(MessagesState):
    company_of_interest: str 
    trade_date: str 
    sender: str 
    # each analyst will populate its own report field 
    market_report: str 
    senfiment_report: str 
    news_report: str 
    fundamentals_report: str 
    # nested states for the debates 
    investment_debate_state: InvestDebateState
    invesment_plan: str 
    trader_investment_plan: str 
    risk_debate_state: RiskDebateState
    final_trade_decision: str  

### Tools

In [10]:
import yfinance as yf 
from langchain_core.tools import tool

@tool
def get_yfinance_data(
    symbol: Annotated[str, "ticker symbol of the company"],
    start_date: Annotated[str, "Start date in yyyy-mm-dd format"],
    end_date: Annotated[str, "End date in yyyy-mm-dd format"]
) -> str:
    """
    Retrieve the stock price data for a given ticker symbol from Yahoo Finance 
    """
    try:
        ticker = yf.Ticker(symbol.upper())
        data = ticker.history(start=start_date, end=end_date)
        if data.empty:
            return f"No data found for symbol '{symbol}' between {start_date} and {end_date}"
        return data.to_csv()
    except Exception as e:
        return f"Error fetching Yahoo Finance data: {e}"

ModuleNotFoundError: No module named 'yfinance'

In [ ]:
from stockstats import wrap as stockstats_wrap

@tool
def get_technical_indicators(
    symbol: Annotated[str, "ticker symbol of the company"],
    start_date: Annotated[str, "start date in yyyy-mm-dd format"],
    end_date: Annotated[str, "End date in yyyy-mm-dd format"]
) -> str:
    """
    Retrieve key technical indicators for a stock using stockstats library 
    """
    try:
        df = yf.download(symbol, start=start_date, end=end_date, progress=False)
        if df.empty:
            return "No data to calculate indicators"
        stock_df = stockstats_wrap(df)
        indicators = stock_df[['macd', 'rsi_14', 'boll', 'boll_ub', 'boll_lb', 'close_50_sma', 'close_200_sma']]